# Phase 2: Structural Analysis of LSC Circuits

## Overview
This notebook performs **descriptive structural analysis** of 60 ACDC-discovered circuits
across 4 Pythia models, 5 frequency bands, and 3 draws. We analyze the graph topology of
each circuit: which edges are included, how information flows between layers and components,
which attention heads participate, and how circuits overlap across frequency bands.

## Key Questions
1. **Edge Statistics**: How many edges do circuits have? How does sparsity vary by band/model?
2. **Layer Flow**: How does information flow between layers? How prevalent are skip connections?
3. **Head Connectivity**: Which attention heads are most active? Does participation vary by band?
4. **Circuit Similarity**: How similar are circuits across bands? Are there universal or band-specific edges?
5. **Model Scaling**: How do structural properties change across model sizes?

## Data Sources
- Pre-extracted structural data from `outputs/extraction/all_circuits_structure.json`
- 60 circuits: 4 models x 5 bands x 3 draws
- Models: pythia-70m (6L/8H), pythia-160m (12L/12H), pythia-410m (24L/16H), pythia-1b (16L/8H)
- Bands: low, medium, high, very_high, control

## Notebook Structure
1. Setup & Data Loading
2. Dataset Overview
3. Edge Statistics
4. Layer Flow Analysis
5. Head Connectivity
6. Circuit Similarity (Universal & Band-Specific Edges)
7. Jaccard Similarity
8. Model Scaling
9. Summary & Export

## 1. Setup & Data Loading

In [1]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import defaultdict

# Add project path
sys.path.insert(0, "LSC_circuit_analysis/02_Phase_Structural")

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_BANDS,
    FREQUENCY_RANK,
    MODEL_DIR_NAMES,
    MODEL_LAYERS,
    MODEL_HEADS,
    MODEL_INFO,
    MODEL_CAPACITY,
    MODEL_TOTAL_EDGES,
    MODEL_THRESHOLDS,
    BAND_COLORS,
    MODEL_COLORS,
    BAND_NAMES,
    COMPONENT_COLORS,
    EDGE_CATEGORY_COLORS,
    SOURCE_TYPES,
    DESTINATION_TYPES,
    COMPONENT_TYPES,
    ANALYSIS_DIR,
    VIZ_DIR,
    get_output_dirs,
)
from utils.data_loading import load_extracted_data
from utils.edge_analysis import (
    get_edge_set,
    compute_jaccard,
    compute_containment,
    compute_universal_edges_per_draw,
    compute_universal_edges_across_draws,
    compute_band_specific_edges,
    compute_edge_sharing_spectrum,
    compute_jaccard_matrix,
    compute_band_jaccard_summary,
    compute_within_between_jaccard,
)
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_layer_flow_heatmap,
    plot_head_participation_heatmap,
    plot_jaccard_heatmap,
    plot_band_jaccard_heatmap,
)

# Setup
setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()
print(f"Analysis output: {ANALYSIS_DIR}")
print(f"Visualization output: {VIZ_DIR}")

Analysis output: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis
Visualization output: LSC_circuit_analysis/02_Phase_Structural/outputs/viz


In [2]:
# Load extracted data
circuits, df = load_extracted_data()
print(f"\nDataFrame shape: {df.shape}")
print(f"Models: {df['model'].unique().tolist()}")
print(f"Bands: {df['band'].unique().tolist()}")
print(f"Draws: {df['draw'].unique().tolist()}")
df.head()

Loaded 75 circuits (0 failed)

DataFrame shape: (75, 30)
Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']


,circuit_id,model,band,draw,threshold,total_edges,total_possible_edges,edge_fraction,n_skip,n_input,...,comp_resid_post,attn_fraction,mlp_fraction,resid_fraction,active_heads,total_heads,head_participation_rate,mean_edges_per_head,max_edges_per_head,frequency_rank
0,pythia_70m_low_draw_1,pythia-70m,low,draw_1,0.00158,380,1324,0.287009,235,17,...,50,0.544737,0.323684,0.131579,39,48,0.812500,5.307692,20,1.0
1,pythia_70m_low_draw_2,pythia-70m,low,draw_2,0.00158,396,1324,0.299094,249,17,...,50,0.555556,0.318182,0.126263,40,48,0.833333,5.500000,22,1.0
2,pythia_70m_low_draw_3,pythia-70m,low,draw_3,0.00158,391,1324,0.295317,244,17,...,51,0.544757,0.324808,0.130435,41,48,0.854167,5.195122,21,1.0
3,pythia_70m_medium_draw_1,pythia-70m,medium,draw_1,0.00158,395,1324,0.298338,248,18,...,50,0.562025,0.311392,0.126582,40,48,0.833333,5.550000,22,2.0
4,pythia_70m_medium_draw_2,pythia-70m,medium,draw_2,0.00158,392,1324,0.296073,244,17,...,50,0.551020,0.321429,0.127551,40,48,0.833333,5.400000,22,2.0


## 2. Dataset Overview

In [3]:
print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"Total circuits: {len(df)}")
print(f"Models: {len(MODELS)} ({MODELS})")
print(f"Bands: {len(BANDS)} ({BANDS})")
print(f"Draws: {len(DRAWS)} ({DRAWS})")
print()
print("Model architectures:")
for model in MODELS:
    info = MODEL_INFO[model]
    thresh = MODEL_THRESHOLDS[model]
    total = MODEL_TOTAL_EDGES[model]
    print(
        f"  {model}: {info['n_layers']}L / {info['n_heads']}H / "
        f"d_model={info['d_model']} / threshold={thresh} / "
        f"total_possible={total}"
    )
print()
print("Per-model summary:")
for model in MODELS:
    subset = df[df["model"] == model]
    print(
        f"  {model}: edges={subset['total_edges'].mean():.0f} +/- {subset['total_edges'].std():.0f}, "
        f"fraction={subset['edge_fraction'].mean():.1%} +/- {subset['edge_fraction'].std():.1%}"
    )

DATASET OVERVIEW
Total circuits: 75
Models: 5 (['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b'])
Bands: 5 (['low', 'medium', 'high', 'very_high', 'control'])
Draws: 3 (['draw_1', 'draw_2', 'draw_3'])

Model architectures:
  pythia-70m: 6L / 8H / d_model=512 / threshold=0.00158 / total_possible=1324
  pythia-160m: 12L / 12H / d_model=768 / threshold=0.000631 / total_possible=11467
  pythia-410m: 24L / 16H / d_model=1024 / threshold=0.000251 / total_possible=80581
  pythia-1b: 16L / 8H / d_model=2048 / threshold=0.00158 / total_possible=10009
  pythia-1.4b: 24L / 16H / d_model=2048 / threshold=0.000631 / total_possible=80581

Per-model summary:
  pythia-70m: edges=409 +/- 16, fraction=30.9% +/- 1.2%
  pythia-160m: edges=1407 +/- 62, fraction=12.3% +/- 0.5%
  pythia-410m: edges=3581 +/- 291, fraction=4.4% +/- 0.4%
  pythia-1b: edges=915 +/- 64, fraction=9.1% +/- 0.6%
  pythia-1.4b: edges=2277 +/- 227, fraction=2.8% +/- 0.3%


## 3. Edge Statistics

### 3.1 Total Edges & Edge Fraction by Model x Band

In [4]:
# Edge statistics by model x band
edge_stats = (
    df.groupby(["model", "band"])
    .agg(
        total_edges_mean=("total_edges", "mean"),
        total_edges_std=("total_edges", "std"),
        edge_fraction_mean=("edge_fraction", "mean"),
        edge_fraction_std=("edge_fraction", "std"),
        n=("total_edges", "count"),
    )
    .reset_index()
)

print("Edge Statistics by Model x Band:")
print(edge_stats.to_string(index=False))

edge_stats.to_csv(ANALYSIS_DIR / "edge_stats.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'edge_stats.csv'}")

Edge Statistics by Model x Band:
      model      band  total_edges_mean  total_edges_std  edge_fraction_mean  edge_fraction_std  n
 pythia-70m       low        389.000000         8.185353            0.293807           0.006182  3
 pythia-70m    medium        397.666667         7.371115            0.300352           0.005567  3
 pythia-70m      high        416.666667        13.203535            0.314703           0.009972  3
 pythia-70m very_high        419.333333         4.041452            0.316717           0.003052  3
 pythia-70m   control        423.666667        12.662280            0.319990           0.009564  3
pythia-160m       low       1451.333333        38.837267            0.126566           0.003387  3
pythia-160m    medium       1478.000000        33.600595            0.128892           0.002930  3
pythia-160m      high       1399.666667        48.686069            0.122060           0.004246  3
pythia-160m very_high       1331.333333        26.025628            0.116101

In [5]:
# VIZ 01: Edge fraction by model x band
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Total edges
ax = axes[0]
for model in MODELS:
    subset = df[df["model"] == model]
    means = subset.groupby("band")["total_edges"].mean()
    stds = subset.groupby("band")["total_edges"].std()
    x = range(len(BANDS))
    ax.errorbar(
        x,
        [means.get(b, 0) for b in BANDS],
        yerr=[stds.get(b, 0) for b in BANDS],
        marker="o",
        label=model,
        color=MODEL_COLORS[model],
        capsize=3,
    )
ax.set_xticks(range(len(BANDS)))
ax.set_xticklabels([BAND_NAMES[b] for b in BANDS], rotation=30, ha="right")
ax.set_ylabel("Total Edges")
ax.set_title("Total Edges by Frequency Band")
ax.legend()

# Panel B: Edge fraction
ax = axes[1]
for model in MODELS:
    subset = df[df["model"] == model]
    means = subset.groupby("band")["edge_fraction"].mean()
    stds = subset.groupby("band")["edge_fraction"].std()
    x = range(len(BANDS))
    ax.errorbar(
        x,
        [means.get(b, 0) for b in BANDS],
        yerr=[stds.get(b, 0) for b in BANDS],
        marker="o",
        label=model,
        color=MODEL_COLORS[model],
        capsize=3,
    )
ax.set_xticks(range(len(BANDS)))
ax.set_xticklabels([BAND_NAMES[b] for b in BANDS], rotation=30, ha="right")
ax.set_ylabel("Edge Fraction")
ax.set_title("Circuit Sparsity by Frequency Band")
ax.legend()

fig.suptitle("Circuit Size Across Models and Frequency Bands", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_01_edge_statistics.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_01_edge_statistics.png


### 3.2 Component Type Breakdown

In [6]:
# Component breakdown by model x band
comp_cols = ["attn_fraction", "mlp_fraction", "resid_fraction"]
comp_stats = df.groupby(["model", "band"])[comp_cols].agg(["mean", "std"]).reset_index()
comp_stats.columns = ["_".join(c).rstrip("_") for c in comp_stats.columns]

print("Component Fractions by Model x Band:")
for model in MODELS:
    subset = df[df["model"] == model]
    print(f"\n  {model}:")
    for band in BANDS:
        b = subset[subset["band"] == band]
        if len(b) > 0:
            print(
                f"    {BAND_NAMES[band]:>12}: attn={b['attn_fraction'].mean():.1%}  "
                f"mlp={b['mlp_fraction'].mean():.1%}  resid={b['resid_fraction'].mean():.1%}"
            )

comp_stats.to_csv(ANALYSIS_DIR / "component_breakdown.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'component_breakdown.csv'}")

Component Fractions by Model x Band:

  pythia-70m:
             Low: attn=54.8%  mlp=32.2%  resid=12.9%
          Medium: attn=56.1%  mlp=31.2%  resid=12.7%
            High: attn=58.2%  mlp=29.4%  resid=12.4%
       Very High: attn=58.4%  mlp=29.2%  resid=12.4%
         Control: attn=58.4%  mlp=29.3%  resid=12.3%

  pythia-160m:
             Low: attn=56.9%  mlp=34.4%  resid=8.7%
          Medium: attn=57.9%  mlp=33.6%  resid=8.5%
            High: attn=56.1%  mlp=34.9%  resid=9.0%
       Very High: attn=55.3%  mlp=35.4%  resid=9.3%
         Control: attn=54.9%  mlp=35.9%  resid=9.2%

  pythia-410m:
             Low: attn=57.5%  mlp=35.6%  resid=6.9%
          Medium: attn=56.8%  mlp=36.3%  resid=6.9%
            High: attn=55.9%  mlp=36.6%  resid=7.5%
       Very High: attn=53.6%  mlp=38.4%  resid=8.1%
         Control: attn=54.2%  mlp=38.0%  resid=7.8%

  pythia-1b:
             Low: attn=61.9%  mlp=31.2%  resid=6.9%
          Medium: attn=61.6%  mlp=31.2%  resid=7.2%
            H

In [7]:
# VIZ 02: Stacked component breakdown
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 5), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    subset = df[df["model"] == model]

    band_means = subset.groupby("band")[comp_cols].mean()
    x = range(len(BANDS))

    bottom = np.zeros(len(BANDS))
    for comp, color_key in [
        ("attn_fraction", "attn"),
        ("mlp_fraction", "mlp"),
        ("resid_fraction", "resid"),
    ]:
        vals = [band_means.loc[b, comp] if b in band_means.index else 0 for b in BANDS]
        ax.bar(
            x,
            vals,
            bottom=bottom,
            label=comp.replace("_fraction", ""),
            color=COMPONENT_COLORS[color_key],
            alpha=0.8,
        )
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(
        [BAND_NAMES[b] for b in BANDS], rotation=45, ha="right", fontsize=8
    )
    ax.set_title(model)
    ax.set_ylim(0, 1.05)
    if idx == 0:
        ax.set_ylabel("Fraction of Edges")
    if idx == len(MODELS) - 1:
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

fig.suptitle("Component Type Breakdown by Model and Band", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_02_component_breakdown.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_02_component_breakdown.png


## 4. Layer Flow Analysis

### 4.1 Layer-to-Layer Flow Heatmaps

In [8]:
# VIZ 03: Layer flow heatmaps - one per model, averaged across bands and draws
n_models = len(MODELS)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Aggregate flow matrices across all circuits for this model
    agg_flow = defaultdict(lambda: defaultdict(int))
    n_circuits = 0
    for c in circuits.values():
        if c["model"] == model:
            flow = c.get("layer_flow", {}).get("flow", {})
            for src_str, dst_dict in flow.items():
                for dst_str, count in dst_dict.items():
                    agg_flow[src_str][dst_str] += count
            n_circuits += 1

    # Average
    avg_flow = {
        k: {kk: vv / n_circuits for kk, vv in v.items()} for k, v in agg_flow.items()
    }

    plot_layer_flow_heatmap(
        avg_flow,
        model,
        n_layers,
        title=f"Mean Layer Flow: {model} (avg over {n_circuits} circuits)",
        ax=ax,
    )

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Layer-to-Layer Information Flow", fontsize=16, y=1.01)
fig.tight_layout()
save_figure(fig, "viz_03_layer_flow_heatmaps.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_03_layer_flow_heatmaps.png


### 4.2 Edge Distribution Across Layers

In [9]:
# Compute per-layer edge counts from by_layer data
layer_rows = []
for c in circuits.values():
    if c.get("status") != "success":
        continue
    by_layer = c.get("by_layer", {})
    n_layers = MODEL_INFO[c["model"]]["n_layers"]
    for layer_idx in range(n_layers):
        layer_data = by_layer.get(str(layer_idx), {})
        total_edges = c["total_edges"]
        layer_rows.append(
            {
                "model": c["model"],
                "band": c["band"],
                "draw": c["draw"],
                "layer": layer_idx,
                "layer_frac": layer_idx / (n_layers - 1) if n_layers > 1 else 0,
                "edges": layer_data.get("total", 0),
                "edge_fraction": layer_data.get("total", 0) / total_edges
                if total_edges > 0
                else 0,
                "attn_edges": layer_data.get("attn", 0),
                "mlp_edges": layer_data.get("mlp", 0),
                "resid_edges": layer_data.get("resid", 0),
            }
        )

df_layers = pd.DataFrame(layer_rows)

# Save layer flow stats
layer_stats = (
    df_layers.groupby(["model", "band", "layer"])
    .agg(
        edges_mean=("edges", "mean"),
        edges_std=("edges", "std"),
        edge_fraction_mean=("edge_fraction", "mean"),
    )
    .reset_index()
)
layer_stats.to_csv(ANALYSIS_DIR / "layer_flow_stats.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'layer_flow_stats.csv'}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/layer_flow_stats.csv


In [10]:
# VIZ 04: Edge distribution across layers by band (one panel per model)
n_models = len(MODELS)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    subset = df_layers[df_layers["model"] == model]
    n_layers = MODEL_INFO[model]["n_layers"]

    for band in BANDS:
        band_data = subset[subset["band"] == band]
        means = band_data.groupby("layer")["edge_fraction"].mean()
        ax.plot(
            range(n_layers),
            [means.get(l, 0) for l in range(n_layers)],
            marker="o",
            markersize=3,
            label=BAND_NAMES[band],
            color=BAND_COLORS[band],
            alpha=0.8,
        )

    ax.set_xlabel("Layer")
    ax.set_ylabel("Fraction of Total Edges")
    ax.set_title(f"{model} ({n_layers} layers)")
    ax.legend(fontsize=8)

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Edge Distribution Across Layers", fontsize=14, y=1.01)
fig.tight_layout()
save_figure(fig, "viz_04_layer_edge_distribution.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_04_layer_edge_distribution.png


### 4.3 Edge Category Analysis

In [11]:
# Edge category statistics
cat_cols = [
    "skip_fraction",
    "input_fraction",
    "output_fraction",
    "local_fraction",
    "forward_fraction",
]
cat_stats = df.groupby(["model", "band"])[cat_cols].agg(["mean", "std"]).reset_index()
cat_stats.columns = ["_".join(c).rstrip("_") for c in cat_stats.columns]

print("Edge Category Fractions (mean across draws):")
for model in MODELS:
    subset = df[df["model"] == model]
    print(f"\n  {model}:")
    print(
        f"    {'Band':>12}  {'skip':>7}  {'input':>7}  {'output':>7}  {'local':>7}  {'forward':>7}"
    )
    for band in BANDS:
        b = subset[subset["band"] == band]
        if len(b) > 0:
            print(
                f"    {BAND_NAMES[band]:>12}  "
                f"{b['skip_fraction'].mean():7.1%}  "
                f"{b['input_fraction'].mean():7.1%}  "
                f"{b['output_fraction'].mean():7.1%}  "
                f"{b['local_fraction'].mean():7.1%}  "
                f"{b['forward_fraction'].mean():7.1%}"
            )

cat_stats.to_csv(ANALYSIS_DIR / "edge_category_stats.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'edge_category_stats.csv'}")

Edge Category Fractions (mean across draws):

  pythia-70m:
            Band     skip    input   output    local  forward
             Low    62.4%     4.4%    12.9%     1.9%    35.7%
          Medium    62.9%     4.5%    12.7%     1.8%    35.4%
            High    62.6%     4.6%    12.4%     1.8%    35.6%
       Very High    61.3%     4.3%    12.4%     1.9%    36.8%
         Control    62.1%     4.1%    12.3%     1.8%    36.1%

  pythia-160m:
            Band     skip    input   output    local  forward
             Low    80.3%     1.1%     8.7%     0.7%    19.0%
          Medium    81.0%     1.2%     8.5%     0.7%    18.3%
            High    80.6%     1.1%     9.0%     0.6%    18.8%
       Very High    78.6%     1.3%     9.3%     0.7%    20.8%
         Control    80.5%     1.4%     9.2%     0.7%    18.9%

  pythia-410m:
            Band     skip    input   output    local  forward
             Low    88.7%     0.4%     6.9%     0.2%    11.1%
          Medium    88.7%     0.3%     6

In [12]:
# VIZ 05: Edge categories by model (stacked bar, averaged across bands)
fig, ax = plt.subplots(figsize=(10, 6))

categories = ["skip", "forward", "local", "input", "output"]
x = np.arange(len(MODELS))
width = 0.6

bottom = np.zeros(len(MODELS))
for cat in categories:
    vals = [df[df["model"] == m][f"{cat}_fraction"].mean() for m in MODELS]
    ax.bar(
        x,
        vals,
        width,
        bottom=bottom,
        label=cat.capitalize(),
        color=EDGE_CATEGORY_COLORS[cat],
        alpha=0.85,
    )
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(MODELS)
ax.set_ylabel("Fraction of Edges")
ax.set_title("Edge Category Distribution by Model")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_ylim(0, 1.05)

fig.tight_layout()
save_figure(fig, "viz_05_edge_categories.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_05_edge_categories.png


## 5. Head Connectivity

### 5.1 Head Participation Heatmaps

In [13]:
# VIZ 06: Head participation heatmaps (averaged across bands/draws): separate per model
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    n_heads = MODEL_INFO[model]["n_heads"]

    # Aggregate head counts
    agg_heads = defaultdict(float)
    n_circuits = 0
    for c in circuits.values():
        if c["model"] == model:
            for hk, count in c.get("by_head", {}).items():
                agg_heads[hk] += count
            n_circuits += 1

    avg_heads = {k: v / n_circuits for k, v in agg_heads.items()}

    fig, ax = plt.subplots(figsize=(max(6, n_layers * 0.55), max(4, n_heads * 0.45)))
    plot_head_participation_heatmap(
        avg_heads,
        model,
        n_layers,
        n_heads,
        title=f"Mean Head Participation: {model}",
        ax=ax,
    )
    fig.tight_layout()
    save_figure(fig, f"viz_06_head_participation_{model}.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_06_head_participation_pythia-70m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_06_head_participation_pythia-160m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_06_head_participation_pythia-410m.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_06_head_participation_pythia-1b.png


  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_06_head_participation_pythia-1.4b.png


### 5.2 Head Participation Statistics

In [14]:
# Head participation rates
head_stats = (
    df.groupby(["model", "band"])
    .agg(
        active_heads_mean=("active_heads", "mean"),
        active_heads_std=("active_heads", "std"),
        total_heads=("total_heads", "first"),
        participation_rate_mean=("head_participation_rate", "mean"),
        participation_rate_std=("head_participation_rate", "std"),
        mean_edges_per_head_mean=("mean_edges_per_head", "mean"),
        max_edges_per_head_mean=("max_edges_per_head", "mean"),
    )
    .reset_index()
)

print("Head Participation by Model x Band:")
for model in MODELS:
    subset = head_stats[head_stats["model"] == model]
    print(f"\n  {model} (total heads: {subset['total_heads'].iloc[0]}):")
    for _, row in subset.iterrows():
        print(
            f"    {BAND_NAMES[row['band']]:>12}: "
            f"active={row['active_heads_mean']:.0f}/{row['total_heads']}, "
            f"rate={row['participation_rate_mean']:.1%}, "
            f"mean_edges={row['mean_edges_per_head_mean']:.1f}, "
            f"max_edges={row['max_edges_per_head_mean']:.0f}"
        )

head_stats.to_csv(ANALYSIS_DIR / "head_participation.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'head_participation.csv'}")

Head Participation by Model x Band:

  pythia-70m (total heads: 48):
             Low: active=40/48, rate=83.3%, mean_edges=5.3, max_edges=21
          Medium: active=41/48, rate=84.7%, mean_edges=5.5, max_edges=22
            High: active=41/48, rate=86.1%, mean_edges=5.9, max_edges=24
       Very High: active=40/48, rate=84.0%, mean_edges=6.1, max_edges=23
         Control: active=42/48, rate=86.8%, mean_edges=5.9, max_edges=22

  pythia-160m (total heads: 144):
             Low: active=97/144, rate=67.6%, mean_edges=8.5, max_edges=39
          Medium: active=99/144, rate=69.0%, mean_edges=8.6, max_edges=42
            High: active=98/144, rate=68.3%, mean_edges=8.0, max_edges=40
       Very High: active=96/144, rate=66.7%, mean_edges=7.7, max_edges=39
         Control: active=100/144, rate=69.4%, mean_edges=7.5, max_edges=41

  pythia-410m (total heads: 384):
             Low: active=226/384, rate=58.8%, mean_edges=10.1, max_edges=73
          Medium: active=217/384, rate=56.5%, mea

In [15]:
# VIZ 07: Head participation rate by band (one panel per model)
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 5), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    subset = df[df["model"] == model]

    positions = []
    data = []
    colors = []
    for i, band in enumerate(BANDS):
        vals = subset[subset["band"] == band]["head_participation_rate"].values
        if len(vals) > 0:
            positions.append(i)
            data.append(vals)
            colors.append(BAND_COLORS[band])

    bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels(
        [BAND_NAMES[b] for b in BANDS], rotation=45, ha="right", fontsize=8
    )
    ax.set_title(model)
    if idx == 0:
        ax.set_ylabel("Head Participation Rate")

fig.suptitle("Fraction of Active Attention Heads", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_07_head_participation_rate.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_07_head_participation_rate.png


## 6. Circuit Similarity: Universal & Band-Specific Edges

### 6.1 Universal Edges

In [16]:
# Universal edges per draw and across draws
universal_rows = []

for model in MODELS:
    per_draw = compute_universal_edges_per_draw(circuits, model)
    across = compute_universal_edges_across_draws(circuits, model)

    # Get mean circuit size for this model
    model_circuits = [c for c in circuits.values() if c["model"] == model]
    mean_edges = np.mean([c["total_edges"] for c in model_circuits])

    for draw, edges in per_draw.items():
        # Get circuit size for this draw (any band, since we want fraction)
        draw_circuits = [c for c in model_circuits if c["draw"] == draw]
        draw_mean_edges = np.mean([c["total_edges"] for c in draw_circuits])
        universal_rows.append(
            {
                "model": model,
                "draw": draw,
                "n_universal": len(edges),
                "mean_circuit_edges": draw_mean_edges,
                "universal_fraction": len(edges) / draw_mean_edges
                if draw_mean_edges > 0
                else 0,
            }
        )

    universal_rows.append(
        {
            "model": model,
            "draw": "across_all",
            "n_universal": len(across),
            "mean_circuit_edges": mean_edges,
            "universal_fraction": len(across) / mean_edges if mean_edges > 0 else 0,
        }
    )

df_universal = pd.DataFrame(universal_rows)
print("Universal Edges:")
print(df_universal.to_string(index=False))

df_universal.to_csv(ANALYSIS_DIR / "universal_edges.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'universal_edges.csv'}")

Universal Edges:
      model       draw  n_universal  mean_circuit_edges  universal_fraction
 pythia-70m     draw_1          297          409.600000            0.725098
 pythia-70m     draw_2          297          404.000000            0.735149
 pythia-70m     draw_3          311          414.200000            0.750845
 pythia-70m across_all          268          409.266667            0.654830
pythia-160m     draw_1          694         1403.800000            0.494372
pythia-160m     draw_2          723         1427.200000            0.506586
pythia-160m     draw_3          723         1388.600000            0.520668
pythia-160m across_all          531         1406.533333            0.377524
pythia-410m     draw_1         1283         3560.400000            0.360353
pythia-410m     draw_2         1301         3566.600000            0.364773
pythia-410m     draw_3         1324         3616.600000            0.366090
pythia-410m across_all          872         3581.200000            0.24

### 6.2 Band-Specific Edges

In [17]:
# Band-specific edges
specific_rows = []

for model in MODELS:
    for draw in DRAWS:
        specific = compute_band_specific_edges(circuits, model, draw)
        # Get circuit sizes
        for band in BANDS:
            n_specific = len(specific.get(band, set()))
            # Get total edges for this circuit
            for c in circuits.values():
                if c["model"] == model and c["draw"] == draw and c["band"] == band:
                    total = c["total_edges"]
                    break
            else:
                total = 0
            specific_rows.append(
                {
                    "model": model,
                    "draw": draw,
                    "band": band,
                    "n_specific": n_specific,
                    "total_edges": total,
                    "specific_fraction": n_specific / total if total > 0 else 0,
                }
            )

df_specific = pd.DataFrame(specific_rows)
print("Band-Specific Edges (mean across draws):")
spec_summary = (
    df_specific.groupby(["model", "band"])
    .agg(
        n_specific_mean=("n_specific", "mean"),
        specific_fraction_mean=("specific_fraction", "mean"),
    )
    .reset_index()
)
print(spec_summary.to_string(index=False))

df_specific.to_csv(ANALYSIS_DIR / "band_specific_edges.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'band_specific_edges.csv'}")

Band-Specific Edges (mean across draws):
      model      band  n_specific_mean  specific_fraction_mean
pythia-1.4b   control       393.333333                0.182357
pythia-1.4b      high       385.666667                0.178921
pythia-1.4b       low       616.000000                0.239897
pythia-1.4b    medium       544.666667                0.218787
pythia-1.4b very_high       376.000000                0.185123
pythia-160m   control       118.666667                0.086595
pythia-160m      high       126.000000                0.089556
pythia-160m       low       128.000000                0.087862
pythia-160m    medium       132.666667                0.089729
pythia-160m very_high       125.333333                0.094105
  pythia-1b   control        89.666667                0.102448
  pythia-1b      high       133.666667                0.139838
  pythia-1b       low       142.333333                0.146964
  pythia-1b    medium       132.000000                0.139722
  pythia-1b ve


Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/band_specific_edges.csv


### 6.3 Edge Sharing Spectrum

In [18]:
# Edge sharing spectrum: how many bands does each edge appear in?
sharing_rows = []

for model in MODELS:
    for draw in DRAWS:
        spectrum = compute_edge_sharing_spectrum(circuits, model, draw)
        total_unique = sum(spectrum.values())
        for n_bands, count in spectrum.items():
            sharing_rows.append(
                {
                    "model": model,
                    "draw": draw,
                    "n_bands": n_bands,
                    "count": count,
                    "fraction": count / total_unique if total_unique > 0 else 0,
                }
            )

df_sharing = pd.DataFrame(sharing_rows)
print("Edge Sharing Spectrum (mean fraction across draws):")
sharing_summary = (
    df_sharing.groupby(["model", "n_bands"])["fraction"].mean().reset_index()
)
for model in MODELS:
    sub = sharing_summary[sharing_summary["model"] == model]
    print(f"\n  {model}:")
    for _, row in sub.iterrows():
        bar = "#" * int(row["fraction"] * 50)
        print(f"    {int(row['n_bands'])} bands: {row['fraction']:.1%}  {bar}")

df_sharing.to_csv(ANALYSIS_DIR / "edge_sharing.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'edge_sharing.csv'}")

Edge Sharing Spectrum (mean fraction across draws):

  pythia-70m:
    1 bands: 14.5%  #######
    2 bands: 10.1%  #####
    3 bands: 8.8%  ####
    4 bands: 10.0%  ####
    5 bands: 56.6%  ############################

  pythia-160m:
    1 bands: 27.0%  #############
    2 bands: 17.5%  ########
    3 bands: 12.7%  ######
    4 bands: 12.2%  ######
    5 bands: 30.6%  ###############

  pythia-410m:
    1 bands: 38.8%  ###################
    2 bands: 19.9%  #########
    3 bands: 13.4%  ######
    4 bands: 9.8%  ####
    5 bands: 18.1%  #########

  pythia-1b:
    1 bands: 34.2%  #################
    2 bands: 19.4%  #########
    3 bands: 14.8%  #######
    4 bands: 11.0%  #####
    5 bands: 20.6%  ##########

  pythia-1.4b:
    1 bands: 45.3%  ######################
    2 bands: 20.6%  ##########
    3 bands: 12.5%  ######
    4 bands: 9.5%  ####
    5 bands: 12.1%  ######

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/edge_sharing.csv


In [19]:
# VIZ 08: Universal + band-specific summary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Universal edge fraction by model
ax = axes[0]
draw_only = df_universal[df_universal["draw"] != "across_all"]
for i, model in enumerate(MODELS):
    vals = draw_only[draw_only["model"] == model]["universal_fraction"].values
    ax.bar(
        i,
        np.mean(vals),
        yerr=np.std(vals) if len(vals) > 1 else 0,
        color=MODEL_COLORS[model],
        alpha=0.8,
        capsize=5,
    )
ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels(MODELS, rotation=15)
ax.set_ylabel("Fraction of Edges")
ax.set_title("Universal Edge Fraction (per draw)")

# Panel B: Band-specific edge fraction by model x band
ax = axes[1]
spec_pivot = (
    df_specific.groupby(["model", "band"])["specific_fraction"].mean().reset_index()
)
for model in MODELS:
    sub = spec_pivot[spec_pivot["model"] == model]
    x = range(len(BANDS))
    vals = [
        sub[sub["band"] == b]["specific_fraction"].values[0]
        if len(sub[sub["band"] == b]) > 0
        else 0
        for b in BANDS
    ]
    ax.plot(x, vals, marker="o", label=model, color=MODEL_COLORS[model])
ax.set_xticks(range(len(BANDS)))
ax.set_xticklabels([BAND_NAMES[b] for b in BANDS], rotation=30, ha="right")
ax.set_ylabel("Fraction of Edges")
ax.set_title("Band-Specific Edge Fraction")
ax.legend(fontsize=8)

fig.suptitle("Universal vs Band-Specific Edges", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_08_universal_specific_edges.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_08_universal_specific_edges.png


In [20]:
# VIZ 09: Edge sharing spectrum
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4), sharey=True)

for idx, model in enumerate(MODELS):
    ax = axes[idx]
    sub = df_sharing[df_sharing["model"] == model]
    means = sub.groupby("n_bands")["fraction"].mean()
    stds = sub.groupby("n_bands")["fraction"].std()

    x = sorted(means.index)
    ax.bar(
        x,
        [means[i] for i in x],
        yerr=[stds.get(i, 0) for i in x],
        color=[plt.cm.viridis(i / len(BANDS)) for i in x],
        alpha=0.8,
        capsize=3,
    )
    ax.set_xlabel("Number of Bands")
    ax.set_title(model)
    if idx == 0:
        ax.set_ylabel("Fraction of Unique Edges")

fig.suptitle(
    "Edge Sharing Spectrum: How Many Bands Does Each Edge Appear In?",
    fontsize=14,
    y=1.02,
)
fig.tight_layout()
save_figure(fig, "viz_09_edge_sharing_spectrum.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_09_edge_sharing_spectrum.png


## 7. Jaccard Similarity

### 7.1 Full Jaccard Similarity Matrices

In [21]:
# VIZ 10: Full Jaccard matrices per model
n_models = len(MODELS)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    matrix, labels = compute_jaccard_matrix(circuits, model)
    plot_jaccard_heatmap(
        matrix, labels, model, title=f"Jaccard Similarity: {model}", ax=ax
    )

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Pairwise Jaccard Similarity Between All Circuits", fontsize=16, y=1.01)
fig.tight_layout()
save_figure(fig, "viz_10_jaccard_matrices.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_10_jaccard_matrices.png


### 7.2 Band-to-Band Mean Jaccard

In [22]:
# Band-to-band mean Jaccard and within vs between comparison
jaccard_rows = []
wb_rows = []

for model in MODELS:
    band_jac = compute_band_jaccard_summary(circuits, model)
    for b1 in BANDS:
        for b2 in BANDS:
            jaccard_rows.append(
                {
                    "model": model,
                    "band_1": b1,
                    "band_2": b2,
                    "mean_jaccard": band_jac[b1][b2],
                }
            )

    wb = compute_within_between_jaccard(circuits, model)
    wb_rows.append(
        {
            "model": model,
            "within_mean": np.mean(wb["within"]) if wb["within"] else np.nan,
            "within_std": np.std(wb["within"]) if wb["within"] else np.nan,
            "between_mean": np.mean(wb["between"]) if wb["between"] else np.nan,
            "between_std": np.std(wb["between"]) if wb["between"] else np.nan,
            "n_within": len(wb["within"]),
            "n_between": len(wb["between"]),
        }
    )

df_jaccard = pd.DataFrame(jaccard_rows)
df_wb = pd.DataFrame(wb_rows)

print("Within-Band vs Between-Band Jaccard Similarity:")
print(df_wb.to_string(index=False))

df_jaccard.to_csv(ANALYSIS_DIR / "band_jaccard.csv", index=False)
df_wb.to_csv(ANALYSIS_DIR / "jaccard_summary.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'band_jaccard.csv'}")
print(f"Saved: {ANALYSIS_DIR / 'jaccard_summary.csv'}")

Within-Band vs Between-Band Jaccard Similarity:
      model  within_mean  within_std  between_mean  between_std  n_within  n_between
 pythia-70m     0.794989    0.021808      0.762580     0.030556        30         90
pythia-160m     0.589080    0.019981      0.556812     0.024308        30         90
pythia-410m     0.446278    0.014745      0.430386     0.016502        30         90
  pythia-1b     0.477929    0.024206      0.465130     0.023998        30         90
pythia-1.4b     0.384778    0.027921      0.366090     0.024119        30         90

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/band_jaccard.csv
Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/jaccard_summary.csv


In [23]:
# VIZ 11: Band-to-band Jaccard heatmaps
n_models = len(MODELS)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    ax = axes_flat[idx]
    band_jac = compute_band_jaccard_summary(circuits, model)
    plot_band_jaccard_heatmap(band_jac, model, ax=ax)

for idx in range(n_models, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Band-to-Band Mean Jaccard Similarity", fontsize=16, y=1.01)
fig.tight_layout()
save_figure(fig, "viz_11_band_jaccard_heatmaps.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_11_band_jaccard_heatmaps.png


## 8. Model Scaling

In [24]:
# Structural metrics across model scales
scaling_metrics = [
    "edge_fraction",
    "skip_fraction",
    "head_participation_rate",
    "attn_fraction",
    "mlp_fraction",
    "resid_fraction",
    "local_fraction",
    "forward_fraction",
]

scaling_rows = []
for model in MODELS:
    subset = df[df["model"] == model]
    row = {"model": model, "model_capacity": MODEL_CAPACITY[model]}
    for metric in scaling_metrics:
        row[f"{metric}_mean"] = subset[metric].mean()
        row[f"{metric}_std"] = subset[metric].std()
    scaling_rows.append(row)

df_scaling = pd.DataFrame(scaling_rows)
print("Structural Metrics by Model Scale:")
print(df_scaling.to_string(index=False))

df_scaling.to_csv(ANALYSIS_DIR / "scaling_structural.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'scaling_structural.csv'}")

Structural Metrics by Model Scale:
      model  model_capacity  edge_fraction_mean  edge_fraction_std  skip_fraction_mean  skip_fraction_std  head_participation_rate_mean  head_participation_rate_std  attn_fraction_mean  attn_fraction_std  mlp_fraction_mean  mlp_fraction_std  resid_fraction_mean  resid_fraction_std  local_fraction_mean  local_fraction_std  forward_fraction_mean  forward_fraction_std
 pythia-70m              70            0.309114           0.012230            0.622334           0.007978                      0.850000                     0.026352            0.572010           0.016840           0.302453          0.014107             0.125537            0.003218             0.018402            0.000939               0.359264              0.007782
pythia-160m             160            0.122659           0.005442            0.801768           0.009521                      0.681944                     0.021356            0.562150           0.014470           0.348492       

In [25]:
# VIZ 12: Key structural metrics vs model size
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_metrics = [
    "edge_fraction",
    "skip_fraction",
    "head_participation_rate",
    "attn_fraction",
]
titles = [
    "Edge Fraction (Sparsity)",
    "Skip Connection Fraction",
    "Head Participation Rate",
    "Attention Edge Fraction",
]

for idx, (metric, title) in enumerate(zip(plot_metrics, titles)):
    ax = axes[idx // 2][idx % 2]
    for model in MODELS:
        subset = df[df["model"] == model]
        x = MODEL_CAPACITY[model]
        y = subset[metric].mean()
        yerr = subset[metric].std()
        ax.errorbar(
            x,
            y,
            yerr=yerr,
            marker="o",
            color=MODEL_COLORS[model],
            label=model,
            capsize=5,
            markersize=8,
        )
    ax.set_xlabel("Model Parameters (M)")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(title)
    ax.set_xscale("log")
    ax.legend(fontsize=8)

fig.suptitle("Structural Properties vs Model Scale", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_12_model_scaling.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_12_model_scaling.png


In [26]:
# VIZ 13: Edge fraction by band for each model (scaling perspective)
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(BANDS))
width = 0.18

for i, model in enumerate(MODELS):
    subset = df[df["model"] == model]
    means = [subset[subset["band"] == b]["edge_fraction"].mean() for b in BANDS]
    stds = [subset[subset["band"] == b]["edge_fraction"].std() for b in BANDS]
    ax.bar(
        x + i * width,
        means,
        width,
        yerr=stds,
        label=model,
        color=MODEL_COLORS[model],
        alpha=0.8,
        capsize=3,
    )

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([BAND_NAMES[b] for b in BANDS])
ax.set_ylabel("Edge Fraction")
ax.set_title("Circuit Sparsity: Model x Band Interaction")
ax.legend()

fig.tight_layout()
save_figure(fig, "viz_13_model_band_interaction.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_13_model_band_interaction.png


## 9. Summary & Export

In [27]:
# Master summary table
summary_rows = []
for model in MODELS:
    for band in BANDS:
        subset = df[(df["model"] == model) & (df["band"] == band)]
        if len(subset) == 0:
            continue
        row = {
            "model": model,
            "band": band,
            "n_draws": len(subset),
            "total_edges_mean": subset["total_edges"].mean(),
            "total_edges_std": subset["total_edges"].std(),
            "edge_fraction_mean": subset["edge_fraction"].mean(),
            "skip_fraction_mean": subset["skip_fraction"].mean(),
            "input_fraction_mean": subset["input_fraction"].mean(),
            "local_fraction_mean": subset["local_fraction"].mean(),
            "forward_fraction_mean": subset["forward_fraction"].mean(),
            "attn_fraction_mean": subset["attn_fraction"].mean(),
            "mlp_fraction_mean": subset["mlp_fraction"].mean(),
            "resid_fraction_mean": subset["resid_fraction"].mean(),
            "head_participation_mean": subset["head_participation_rate"].mean(),
            "active_heads_mean": subset["active_heads"].mean(),
            "mean_edges_per_head_mean": subset["mean_edges_per_head"].mean(),
        }
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("Master Summary (20 model x band combinations):")
print(df_summary.to_string(index=False))

df_summary.to_csv(ANALYSIS_DIR / "master_structural_summary.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'master_structural_summary.csv'}")

Master Summary (20 model x band combinations):
      model      band  n_draws  total_edges_mean  total_edges_std  edge_fraction_mean  skip_fraction_mean  input_fraction_mean  local_fraction_mean  forward_fraction_mean  attn_fraction_mean  mlp_fraction_mean  resid_fraction_mean  head_participation_mean  active_heads_mean  mean_edges_per_head_mean
 pythia-70m       low        3        389.000000         8.185353            0.293807            0.623750             0.043715             0.018853               0.357397            0.548350           0.322225             0.129425                 0.833333          40.000000                  5.334271
 pythia-70m    medium        3        397.666667         7.371115            0.300352            0.628588             0.045245             0.017607               0.353805            0.560670           0.311926             0.127404                 0.847222          40.666667                  5.483333
 pythia-70m      high        3        416.666667  

In [28]:
# Export full structure data
df.to_csv(ANALYSIS_DIR / "full_structure_data.csv", index=False)
print(f"Saved: {ANALYSIS_DIR / 'full_structure_data.csv'}")

# Print summary of all outputs
print("\n" + "=" * 70)
print("ALL OUTPUTS")
print("=" * 70)
print("\nAnalysis CSVs:")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}")
print("\nVisualizations:")
for f in sorted(VIZ_DIR.glob("*.png")):
    print(f"  {f.name}")

Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/full_structure_data.csv

ALL OUTPUTS

Analysis CSVs:
  all_hypothesis_tests.csv
  band_jaccard.csv
  band_specific_edges.csv
  component_breakdown.csv
  deep_all_hypothesis_tests.csv
  deep_band_affinity.csv
  deep_band_signatures.csv
  deep_component_jaccard.csv
  deep_component_wiring.csv
  deep_degree_stats.csv
  deep_directed_containment.csv
  deep_draw_stability.csv
  deep_edge_sharing_raw.csv
  deep_graph_metrics.csv
  deep_head_clusters.csv
  deep_head_entropy.csv
  deep_head_universality.csv
  deep_hub_nodes.csv
  deep_input_edges.csv
  deep_layer_sensitivity.csv
  deep_layer_universal_fraction.csv
  deep_master_summary.csv
  deep_output_edges.csv
  deep_reliable_band_specific.csv
  deep_sharing_by_component.csv
  deep_sharing_profiles.csv
  deep_stability_vs_sharing.csv
  deep_universal_core_connectivity.csv
  draw_stability.csv
  edge_category_stats.csv
  edge_sharing.csv
  edge_stats.csv
  full_structure_data.c

In [29]:
# Key findings summary
print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)

print("\n1. CIRCUIT SPARSITY:")
for model in MODELS:
    sub = df[df["model"] == model]
    print(
        f"   {model}: {sub['edge_fraction'].mean():.1%} of possible edges "
        f"({sub['total_edges'].mean():.0f}/{sub['total_possible_edges'].mean():.0f})"
    )

print("\n2. SKIP CONNECTIONS:")
for model in MODELS:
    sub = df[df["model"] == model]
    print(
        f"   {model}: {sub['skip_fraction'].mean():.1%} of edges are skip connections"
    )

print("\n3. HEAD PARTICIPATION:")
for model in MODELS:
    sub = df[df["model"] == model]
    print(
        f"   {model}: {sub['head_participation_rate'].mean():.1%} of heads active, "
        f"{sub['mean_edges_per_head'].mean():.1f} edges/active head"
    )

print("\n4. CIRCUIT SIMILARITY:")
for _, row in df_wb.iterrows():
    print(
        f"   {row['model']}: within-band={row['within_mean']:.3f}, "
        f"between-band={row['between_mean']:.3f}, "
        f"diff={row['within_mean'] - row['between_mean']:.3f}"
    )

print("\n5. UNIVERSAL EDGES:")
for model in MODELS:
    sub = draw_only[draw_only["model"] == model]
    print(f"   {model}: {sub['universal_fraction'].mean():.1%} of edges are universal")


KEY FINDINGS

1. CIRCUIT SPARSITY:
   pythia-70m: 30.9% of possible edges (409/1324)
   pythia-160m: 12.3% of possible edges (1407/11467)
   pythia-410m: 4.4% of possible edges (3581/80581)
   pythia-1b: 9.1% of possible edges (915/10009)
   pythia-1.4b: 2.8% of possible edges (2277/80581)

2. SKIP CONNECTIONS:
   pythia-70m: 62.2% of edges are skip connections
   pythia-160m: 80.2% of edges are skip connections
   pythia-410m: 88.7% of edges are skip connections
   pythia-1b: 79.2% of edges are skip connections
   pythia-1.4b: 84.4% of edges are skip connections

3. HEAD PARTICIPATION:
   pythia-70m: 85.0% of heads active, 5.7 edges/active head
   pythia-160m: 68.2% of heads active, 8.1 edges/active head
   pythia-410m: 55.9% of heads active, 9.3 edges/active head
   pythia-1b: 62.0% of heads active, 7.0 edges/active head
   pythia-1.4b: 44.2% of heads active, 8.3 edges/active head

4. CIRCUIT SIMILARITY:
   pythia-70m: within-band=0.795, between-band=0.763, diff=0.032
   pythia-160m